# Task0, Text Classification

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import nltk

# Import scikit-learn functions and classes
from sklearn.datasets import fetch_20newsgroups # Import fetch_20newsgroups here
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer

!pip install gensim
# Import Gensim functions and classes
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument

# Import NLTK functions and classes
from nltk.tokenize import word_tokenize

# Download necessary NLTK data packages
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# Load Dataset

In [ ]:
categories = None  # Use all categories
twenty_news = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))
y = twenty_news.target

 # ============= Feature Extraction - CountVectorizer =============

In [ ]:
vectorizer = CountVectorizer()
X_count = vectorizer.fit_transform(twenty_news.data)

CountVectorizer already tokenizes text internally, so explicit tokenization is not required.

In [ ]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_count, y, test_size=0.2, random_state=42)


# Define models

In [ ]:
# Define models
models = {
    "Multinomial Naïve Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machine": SVC(),
    "Decision Tree": DecisionTreeClassifier()
}

In [ ]:
# Train and evaluate models
results = []
print("\nResults for CountVectorizer:")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results.append(("CountVectorizer", name, accuracy))
    print(f"{name} Accuracy: {accuracy}")



Results for CountVectorizer:
Multinomial Naïve Bayes Accuracy: 0.6037135278514589
Logistic Regression Accuracy: 0.6681697612732096
Support Vector Machine Accuracy: 0.14668435013262598
Decision Tree Accuracy: 0.43846153846153846


# ============= Feature Extraction - Word2Vec =============

# Tokenization of Text Data
Word2Vec and doc2vwc do not automatically handle tokenization, so we manually tokenized the text using nltk.word_tokenize().

In [ ]:
# Download necessary NLTK data packages
nltk.download('punkt')
nltk.download('punkt_tab') # Download the punkt_tab resource

tokenized_texts = [word_tokenize(text.lower()) for text in twenty_news.data]
w2v_model = Word2Vec(tokenized_texts, vector_size=100, window=5, min_count=2, workers=4)

# Convert documents to feature vectors
def document_vector(w2v_model, doc):
    words = [word for word in doc if word in w2v_model.wv]
    return np.mean([w2v_model.wv[word] for word in words], axis=0) if words else np.zeros(w2v_model.vector_size)

X_w2v = np.array([document_vector(w2v_model, doc) for doc in tokenized_texts])


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_w2v, y, test_size=0.2, random_state=42)


In [ ]:
# Train and evaluate models
print("\nResults for Word2Vec:")
for name, model in models.items():
    # Apply MinMaxScaler if the model is MultinomialNB
    if name == "Multinomial Naïve Bayes":
        from sklearn.preprocessing import MinMaxScaler # Use MinMaxScaler for [0,1] range
        scaler = MinMaxScaler()
        X_train_scaled = scaler.fit_transform(X_train)  # Scale training data
        X_test_scaled = scaler.transform(X_test)      # Scale testing data using the same scaler
        model.fit(X_train_scaled, y_train)             # Train with scaled data
        y_pred = model.predict(X_test_scaled)          # Predict using scaled data
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results.append(("Word2Vec", name, accuracy))
    print(f"{name} Accuracy: {accuracy}")


Results for Word2Vec:
Multinomial Naïve Bayes Accuracy: 0.20742705570291778
Logistic Regression Accuracy: 0.4554376657824934
Support Vector Machine Accuracy: 0.43474801061007956
Decision Tree Accuracy: 0.2116710875331565


# ============= Feature Extraction - Doc2Vec =============


In [ ]:
tagged_data = [TaggedDocument(words=doc, tags=[i]) for i, doc in enumerate(tokenized_texts)]
d2v_model = Doc2Vec(tagged_data, vector_size=100, window=5, min_count=2, workers=4, epochs=20)
X_d2v = np.array([d2v_model.infer_vector(doc.words) for doc in tagged_data])


In [ ]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_d2v, y, test_size=0.2, random_state=42)

In [ ]:
# Train and evaluate models
print("\nResults for Doc2Vec:")
for name, model in models.items():
    # Apply MinMaxScaler if the model is MultinomialNB
    if name == "Multinomial Naïve Bayes":
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
        X_train_scaled = scaler.fit_transform(X_train)  # Scale training data
        X_test_scaled = scaler.transform(X_test)      # Scale testing data using the same scaler
        model.fit(X_train_scaled, y_train)             # Train with scaled data
        y_pred = model.predict(X_test_scaled)          # Predict using scaled data
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results.append(("Doc2Vec", name, accuracy))
    print(f"{name} Accuracy: {accuracy}")


Results for Doc2Vec:
Multinomial Naïve Bayes Accuracy: 0.16843501326259946
Logistic Regression Accuracy: 0.5681697612732095
Support Vector Machine Accuracy: 0.5631299734748011
Decision Tree Accuracy: 0.20053050397877983


# Save results

In [ ]:
results_df = pd.DataFrame(results, columns=["Feature Extractor", "Algorithm", "Accuracy"])
output_file = "Nahid_Task0_Text_Classification.txt"
with open(output_file, "w") as f:
    f.write(results_df.to_string(index=False))

print(f"Benchmark results saved to {output_file}")


Benchmark results saved to Nahid_Task0_Text_Classification.txt
